# Study 08: False Positive Reduction & Post-Processing\n**Goal:** Sweep FP filter thresholds and test morphological cleanup / CRF to improve segmentation quality.

## 1. Setup

In [ ]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import torch
from PIL import Image
sns.set_theme(style='whitegrid')
warnings.filterwarnings('ignore')
print('Setup complete')

## 2. Load Model & Sample Test Slices

In [ ]:
from src.config import DEVICE
from src.data_loader import DatasetConfig, DataPathManager, VolumeWiseSplitter
from src.models import create_model
from src.metrics import SegmentationMetrics

model = create_model('mobilenetv2_unet', in_channels=1, out_channels=1).to(DEVICE)
ckpt = torch.load(Path.cwd().parent/'models'/'best_model.pth', map_location='cpu', weights_only=True)
if isinstance(ckpt, dict) and 'model_state' in ckpt: ckpt = ckpt['model_state']
model.load_state_dict(ckpt, strict=False)
model.eval()

# Load a subset of test slices
path_manager = DataPathManager()
volume_index = path_manager.build_index()
splitter = VolumeWiseSplitter()
splits = splitter.load_splits(DatasetConfig.SPLITS_DIR)
test_vids = splits['test'][:5]  # subset
pairs = [(vid, idx) for vid in test_vids for idx in range(len(volume_index['image_paths'].get(vid, [])))][:200]
print(f'Testing on {len(pairs)} slices')

## 3. Baseline Predictions

In [ ]:
from src.preprocessing import PreprocessingTransform
transform = PreprocessingTransform((256, 256), -100, 400)
images, masks = [], []
for vid, sid in tqdm(pairs, desc='Loading'):
    imp = volume_index['image_paths'][vid][sid]; msk = volume_index['mask_paths'][vid][sid]
    img = np.array(Image.open(imp).convert('L'), dtype=np.float32)
    m = np.array(Image.open(msk), dtype=np.uint8)
    images.append(img); masks.append(m)

metrics_fn = SegmentationMetrics()
baseline_dices = []
for img, mask in tqdm(zip(images, masks), desc='Inference'):
    img_t = transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = (torch.sigmoid(model(img_t)) > 0.5).cpu().numpy()[0,0].astype(np.uint8)
    baseline_dices.append(metrics_fn(mask, pred)['dice'])
print(f'Baseline Dice: {np.mean(baseline_dices):.4f}\u00b1{np.std(baseline_dices):.4f}')

## 4. FP Threshold Sweep

In [ ]:
from skimage import measure

thresholds = np.arange(0.3, 0.95, 0.05)
min_sizes = [0, 16, 32, 64, 128, 256]
results = []

for thresh in tqdm(thresholds, desc='Threshold sweep'):
    for ms in min_sizes:
        dices = []
        for img, mask in zip(images, masks):
            img_t = transform(img).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                prob = torch.sigmoid(model(img_t)).cpu().numpy()[0,0]
            pred = (prob > thresh).astype(np.uint8)
            if ms > 0:
                labeled = measure.label(pred, connectivity=2)
                for region_id in range(1, labeled.max()+1):
                    if (labeled == region_id).sum() < ms:
                        pred[labeled == region_id] = 0
            dices.append(metrics_fn(mask, pred)['dice'])
        results.append({'threshold': float(thresh), 'min_size': ms,
                        'dice_mean': float(np.mean(dices)), 'dice_std': float(np.std(dices))})

## 5. Sweep Results

In [ ]:
# Heatmap
pivot = np.full((len(thresholds), len(min_sizes)), np.nan)
for r in results:
    i = list(thresholds).index(r['threshold'])
    j = min_sizes.index(r['min_size'])
    pivot[i, j] = r['dice_mean']

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(pivot, cmap='RdYlGn', aspect='auto', vmin=0.8, vmax=1.0)
ax.set_xticks(range(len(min_sizes))); ax.set_xticklabels(min_sizes)
ax.set_yticks(range(len(thresholds))); ax.set_yticklabels([f'{t:.2f}' for t in thresholds])
ax.set_xlabel('Min component size (pixels)'); ax.set_ylabel('Probability threshold')
ax.set_title('Mean Dice by Threshold & Min Component Size')
plt.colorbar(im, label='Dice')
for i in range(len(thresholds)):
    for j in range(len(min_sizes)):
        ax.text(j, i, f'{pivot[i,j]:.3f}', ha='center', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(Path.cwd().parent/'figures'/'fp_sweep_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Best Configuration

In [ ]:
best = max(results, key=lambda x: x['dice_mean'])
print(f'Best config: threshold={best["threshold"]}, min_size={best["min_size"]}')
print(f'  Dice: {best["dice_mean"]:.4f}\u00b1{best["dice_std"]:.4f}')
print(f'  Delta from baseline: {best["dice_mean"] - np.mean(baseline_dices):.4f}')

with open(Path.cwd().parent/'results'/'fp_reduction.json', 'w') as f:
    json.dump({'best': best, 'all': results, 'baseline': float(np.mean(baseline_dices))}, f, indent=2)
print('Results saved')

## 7. Post-Processing: Morphological Cleanup

In [ ]:
from scipy import ndimage as ndi
improved = 0
for idx, (img, mask) in enumerate(tqdm(zip(images, masks), desc='Cleanup')):
    img_t = transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        prob = torch.sigmoid(model(img_t)).cpu().numpy()[0,0]
    pred = (prob > best['threshold']).astype(np.uint8)
    # Remove small components
    if best['min_size'] > 0:
        labeled = measure.label(pred, connectivity=2)
        for rid in range(1, labeled.max()+1):
            if (labeled == rid).sum() < best['min_size']:
                pred[labeled == rid] = 0
    # Morphological closing
    cleaned = ndi.binary_closing(pred, structure=np.ones((3,3))).astype(np.uint8)
    d_before = metrics_fn(mask, pred)['dice']
    d_after = metrics_fn(mask, cleaned)['dice']
    if d_after > d_before: improved += 1

print(f'Morphological cleanup improved {improved}/{len(masks)} slices')

## 8. Summary

In [ ]:
print('='*60)
print('FP REDUCTION & POST-PROCESSING SUMMARY')
print('='*60)
print(f'Baseline Dice: {np.mean(baseline_dices):.4f}')
print(f'Best FP config: thresh={best["threshold"]}, min_size={best["min_size"]}')
print(f'  -> Dice: {best["dice_mean"]:.4f}')
print(f'Morphological cleanup helped {improved}/{len(masks)} slices')
print(f'\nNote: If baseline Dice is high (~0.97) and no improvement, model already overfits to background')